# 07. Packaging a Repeatable Processing Script
**Engineering Data Processing with Python** | one-day course | approx. 45 minutes

> **SOLUTION NOTEBOOK.** Every blank and TODO is filled in. Try the exercise version first; use this when you are stuck or to compare approaches. There is usually more than one correct answer.


In [1]:
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## Why bother?

A notebook is a good place to work things out. It is a poor thing to hand to a colleague and say "run this every
month". Cells can run out of order, settings are scattered through the code, and the logic is mixed up with
exploration.

A **script** with three properties fixes that:

1. **Functions with clear inputs and outputs.** Each step (load, clean, dedupe, join, validate, save) is a function
   you can test on its own.
2. **Configuration separate from code.** File paths, column names, mappings and thresholds live in a config file.
   A new export with different column names means editing the config, not the code.
3. **Same input, same output, every time.** Plus a log of what happened.

Today's script reuses what you built in Modules 03 to 06. Very little in it is new.

```
project/
├── pipeline.py          # the code (rarely changes)
├── config_cmms.json     # settings for one data source
├── config_galway.json   # settings for another (Module 08)
├── data/                # raw inputs, never edited
└── outputs/<run_name>/  # one folder per run: clean data, issues, report, log
```

## 1. The configuration

### Exercise 7.1 [Level 1: fill the blanks]
A Python dictionary that we save as JSON. JSON is the same idea as the boiler data dictionary from the primer.

In [2]:
import json

CONFIG = {
    "run_name": "cmms_full_history",
    "source": {
        "path": "data/work_orders_cmms.csv",
        "sep": ",",
        "decimal": ".",
        "date_formats": ["%d/%m/%Y %H:%M", "%Y-%m-%d %H:%M"],
        "modified_formats": ["%Y-%m-%d %H:%M:%S"],
        "column_map": {},                      # CMMS column names already match ours
    },
    "na_values": ["", "N/A", "-", "TBC"],
    "type_map": {"PM": "PM", "PREVENTIVE": "PM", "CM": "CM", "CORRECTIVE": "CM", "BREAKDOWN": "CM",
                 "INSP": "INSP", "INSPECTION": "INSP"},
    "staff": ["Aoife Byrne", "Ciarán Walsh", "Declan Murphy", "Niamh O'Connor",
              "Seán Kelly", "Orla Doyle", "Patrick Ryan", "Gráinne Fitzgerald"],
    "technician_map_file": "data/checkpoints/m04_technician_map.csv",
    "asset_register_file": "data/checkpoints/m04_asset_register_dedup.csv",
    "decommissioned_file": "data/asset_register.xlsx",
    "thresholds": {"max_hours": 80, "max_cost": 50000, "open_days": 60, "as_of": "2026-09-30"},
    "error_rules": ["closed_before_raised", "negative_hours", "implausible_hours",
                    "closed_without_date", "unknown_asset", "implausible_cost"],
    "output_dir": "outputs",
}

with open("config_cmms.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)
print(open("config_cmms.json", encoding="utf-8").read()[:600])

{
  "run_name": "cmms_full_history",
  "source": {
    "path": "data/work_orders_cmms.csv",
    "sep": ",",
    "decimal": ".",
    "date_formats": [
      "%d/%m/%Y %H:%M",
      "%Y-%m-%d %H:%M"
    ],
    "modified_formats": [
      "%Y-%m-%d %H:%M:%S"
    ],
    "column_map": {}
  },
  "na_values": [
    "",
    "N/A",
    "-",
    "TBC"
  ],
  "type_map": {
    "PM": "PM",
    "PREVENTIVE": "PM",
    "CM": "CM",
    "CORRECTIVE": "CM",
    "BREAKDOWN": "CM",
    "INSP": "INSP",
    "INSPECTION": "INSP"
  },
  "staff": [
    "Aoife Byrne",
    "Ciarán Walsh",
    "Declan Murphy",
    "Niam


## 2. The script

The next cell writes `pipeline.py` (`%%writefile` turns the cell's contents into a file instead of running it).
In the exercise version three pieces are blank: the keep-latest dedup, the hours threshold read from config, and two
steps in `run_pipeline`.

In [3]:
%%writefile pipeline.py
"""
Kilbrack work order processing pipeline.

Usage:   python pipeline.py config_cmms.json

Everything that changes between data sources (file path, separator, column names, date formats, category
mappings, thresholds) lives in the JSON config. The code below should not need editing for a new export.
"""
import difflib
import json
import logging
import sys
import unicodedata
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

log = logging.getLogger("pipeline")

REQUIRED = ["wo_id", "asset_id", "wo_type", "priority", "date_raised", "date_closed", "status",
            "labour_hours", "parts_cost_eur", "technician", "description", "last_modified"]


# ------------------------------------------------------------------ small helpers (each one testable on its own)
def to_number(series, decimal="."):
    """Text to float. Handles '€1,250.00' (decimal='.') and '1.250,00' (decimal=',')."""
    s = series.astype("string").str.replace("€", "", regex=False).str.replace(" ", "", regex=False)
    if decimal == ",":
        s = s.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    else:
        s = s.str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")


def parse_dates(series, formats):
    """Try each format in turn; values that fit none stay NaT."""
    result = pd.Series(pd.NaT, index=series.index, dtype="datetime64[ns]")
    for fmt in formats:
        result = result.fillna(pd.to_datetime(series, format=fmt, errors="coerce"))
    return result


def normalise_asset_id(series):
    s = series.astype("string").str.strip().str.upper()
    return s.str.replace(r"^([A-Z]+)-?(\d+)$", r"\1-\2", regex=True)


def normalise_name(name):
    if pd.isna(name):
        return name
    text = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    text = text.replace("'", "").replace(".", " ")
    if "," in text:
        last, first = [p.strip() for p in text.split(",", 1)]
        text = f"{first} {last}"
    return " ".join(text.lower().split())


# ------------------------------------------------------------------ pipeline steps
def load_work_orders(cfg):
    src = cfg["source"]
    df = pd.read_csv(src["path"], sep=src.get("sep", ","), dtype=str, keep_default_na=False)
    log.info(f"Loaded {len(df)} rows from {src['path']}")
    df = df.rename(columns=src.get("column_map", {}))
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"Columns missing after renaming: {missing}. Check source.column_map in the config.")
    df = df[REQUIRED].replace({v: np.nan for v in cfg["na_values"]})
    return df


def clean_work_orders(df, cfg):
    src = cfg["source"]
    df = df.copy()
    df["asset_id"] = normalise_asset_id(df["asset_id"])
    df["status"] = df["status"].str.strip().str.title()

    # Categories: anything unmapped stops the run, with the offending values in the message
    raw_types = df["wo_type"].str.strip().str.upper()
    df["wo_type"] = raw_types.map(cfg["type_map"])
    unmapped = sorted(raw_types[df["wo_type"].isna()].dropna().unique())
    if unmapped:
        raise ValueError(f"Unmapped work order types {unmapped}. Add them to type_map in the config.")

    p = df["priority"].astype("string").str.strip().str.upper()
    p = p.str.replace("P", "", regex=False).str.replace("URGENT", "1", regex=False)
    df["priority"] = pd.to_numeric(p, errors="coerce").astype("Int64")

    for col in ["labour_hours", "parts_cost_eur"]:
        df[col] = to_number(df[col], src.get("decimal", "."))
    df["hours_missing"] = df["labour_hours"].isna()

    for col in ["date_raised", "date_closed"]:
        df[col] = parse_dates(df[col], src["date_formats"])
    df["last_modified"] = parse_dates(df["last_modified"], src["modified_formats"])
    if df["date_raised"].isna().any():
        bad = df.loc[df["date_raised"].isna(), "wo_id"].tolist()[:5]
        raise ValueError(f"date_raised did not parse for e.g. {bad}. Check source.date_formats in the config.")

    # Technicians: approved mapping first, fuzzy match against the staff list as a fallback
    approved = pd.read_csv(cfg["technician_map_file"])
    name_map = dict(zip(approved["tech_norm"], approved["technician"]))
    staff_by_norm = {normalise_name(s): s for s in cfg["staff"]}
    df["technician_raw"] = df["technician"]
    norm = df["technician"].apply(normalise_name)

    def match(n):
        if pd.isna(n):
            return np.nan
        if n in name_map:
            return name_map[n]
        hit = difflib.get_close_matches(n, staff_by_norm.keys(), n=1, cutoff=0.85)
        return staff_by_norm[hit[0]] if hit else np.nan

    df["technician"] = norm.apply(match)
    unknown = sorted(df.loc[df["technician"].isna() & df["technician_raw"].notna(), "technician_raw"].unique())
    if unknown:
        log.warning(f"Technician names needing a human: {unknown}")
    return df


def dedupe_work_orders(df):
    n0 = len(df)
    df = df.drop_duplicates()
    n1 = len(df)
    # Same work order exported more than once: keep the most recently modified version
    df = df.sort_values("last_modified").drop_duplicates(subset="wo_id", keep="last")
    log.info(f"Dedup: {n0 - n1} exact duplicates, {n1 - len(df)} older versions removed, {len(df)} work orders")
    if not df["wo_id"].is_unique:
        raise ValueError("wo_id still not unique after dedup")
    return df.sort_values("date_raised").reset_index(drop=True)


def join_assets(df, cfg):
    reg = pd.read_csv(cfg["asset_register_file"])
    decom = pd.read_excel(cfg["decommissioned_file"], sheet_name="Decommissioned")
    in_service = reg[["asset_id", "asset_type", "site", "criticality"]].assign(asset_status="In Service")
    retired = (decom.rename(columns={"Asset ID": "asset_id", "Asset Type": "asset_type", "Site": "site",
                                     "Criticality": "criticality"})
                    [["asset_id", "asset_type", "site", "criticality"]].assign(asset_status="Decommissioned"))
    assets = pd.concat([in_service, retired], ignore_index=True)
    n = len(df)
    out = df.merge(assets, on="asset_id", how="left", validate="many_to_one", indicator=True)
    out.loc[out["_merge"] == "left_only", "asset_status"] = "Unknown asset"
    assert len(out) == n, "Join changed the number of rows"
    return out.drop(columns="_merge")


def validate(df, cfg):
    t = cfg["thresholds"]
    as_of = pd.Timestamp(t["as_of"])
    rules = {
        "closed_before_raised": df["date_closed"] < df["date_raised"],
        "negative_hours": df["labour_hours"] < 0,
        "implausible_hours": df["labour_hours"] > t["max_hours"],
        "closed_without_date": (df["status"] == "Closed") & df["date_closed"].isna(),
        "unknown_asset": df["asset_status"] == "Unknown asset",
        "implausible_cost": df["parts_cost_eur"] > t["max_cost"],
        "negative_cost": df["parts_cost_eur"] < 0,
        "missing_hours": df["labour_hours"].isna(),
        "missing_cost": df["parts_cost_eur"].isna(),
        "open_too_long": (df["status"] != "Closed") & ((as_of - df["date_raised"]).dt.days > t["open_days"]),
    }
    flags = pd.DataFrame(rules)
    df = df.copy()
    df["dq_issues"] = flags.apply(lambda r: ", ".join(r.index[r]), axis=1)
    df["dq_error"] = flags[cfg["error_rules"]].any(axis=1)
    report = pd.DataFrame({"rule": flags.columns,
                           "severity": ["error" if r in cfg["error_rules"] else "warning" for r in flags.columns],
                           "rows_flagged": flags.sum().values})
    report["pct"] = (report["rows_flagged"] / len(df) * 100).round(2)
    for _, r in report[report["rows_flagged"] > 0].iterrows():
        (log.warning if r["severity"] == "error" else log.info)(f"Rule {r['rule']}: {r['rows_flagged']} rows")
    return df, report


def save_outputs(df, report, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_dir / "work_orders_clean.csv", index=False)
    df[~df["dq_error"]].to_csv(out_dir / "work_orders_kpi_ready.csv", index=False)
    df[df["dq_issues"] != ""].to_csv(out_dir / "dq_issues.csv", index=False)
    report.to_csv(out_dir / "dq_report.csv", index=False)
    log.info(f"Outputs written to {out_dir}")


def run_pipeline(cfg):
    out_dir = Path(cfg["output_dir"]) / cfg["run_name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)-7s %(message)s", force=True,
                        handlers=[logging.FileHandler(out_dir / "pipeline.log", mode="w"),
                                  logging.StreamHandler(sys.stdout)])
    log.info(f"Run '{cfg['run_name']}' started")
    df = load_work_orders(cfg)
    df = clean_work_orders(df, cfg)
    df = dedupe_work_orders(df)
    df = join_assets(df, cfg)
    df, report = validate(df, cfg)
    save_outputs(df, report, out_dir)
    summary = {"rows_out": len(df), "errors": int(df["dq_error"].sum()), "output_dir": str(out_dir)}
    log.info(f"Finished: {summary}")
    return summary


if __name__ == "__main__":
    if len(sys.argv) != 2:
        sys.exit("Usage: python pipeline.py <config.json>")
    with open(sys.argv[1], encoding="utf-8") as f:
        config = json.load(f)
    run_pipeline(config)

Writing pipeline.py


## 3. Run it

`!` runs a command as if you typed it in a terminal. `{sys.executable}` is "the Python this notebook uses", so this
line is the same as typing `python pipeline.py config_cmms.json` on your own machine.

In [4]:
import sys
!{sys.executable} pipeline.py config_cmms.json

2026-10-06 14:14:37,661 INFO    Run 'cmms_full_history' started
2026-10-06 14:14:37,666 INFO    Loaded 1016 rows from data/work_orders_cmms.csv


2026-10-06 14:14:37,688 INFO    Dedup: 18 exact duplicates, 25 older versions removed, 973 work orders


2026-10-06 14:14:37,910 WARNING Rule closed_before_raised: 6 rows
2026-10-06 14:14:37,910 WARNING Rule negative_hours: 4 rows
2026-10-06 14:14:37,911 WARNING Rule implausible_hours: 3 rows
2026-10-06 14:14:37,911 WARNING Rule closed_without_date: 5 rows
2026-10-06 14:14:37,911 WARNING Rule unknown_asset: 6 rows
2026-10-06 14:14:37,912 INFO    Rule negative_cost: 1 rows
2026-10-06 14:14:37,912 INFO    Rule missing_hours: 33 rows
2026-10-06 14:14:37,912 INFO    Rule missing_cost: 88 rows


2026-10-06 14:14:37,960 INFO    Outputs written to outputs/cmms_full_history
2026-10-06 14:14:37,960 INFO    Finished: {'rows_out': 973, 'errors': 24, 'output_dir': 'outputs/cmms_full_history'}


In [5]:
from pathlib import Path
run_dir = Path("outputs/cmms_full_history")
print(sorted(p.name for p in run_dir.iterdir()))
pd.read_csv(run_dir / "dq_report.csv")

['dq_issues.csv', 'dq_report.csv', 'pipeline.log', 'work_orders_clean.csv', 'work_orders_kpi_ready.csv']


,rule,severity,rows_flagged,pct
0,closed_before_raised,error,6,0.62
1,negative_hours,error,4,0.41
2,implausible_hours,error,3,0.31
3,closed_without_date,error,5,0.51
4,unknown_asset,error,6,0.62
5,implausible_cost,error,0,0.00
6,negative_cost,warning,1,0.10
7,missing_hours,warning,33,3.39
8,missing_cost,warning,88,9.04
9,open_too_long,warning,0,0.00


In [6]:
files = {p.name for p in run_dir.iterdir()} if run_dir.exists() else set()
check({"work_orders_clean.csv", "work_orders_kpi_ready.csv", "dq_issues.csv", "dq_report.csv", "pipeline.log"} <= files,
      "The pipeline ran end to end and wrote all its outputs.",
      "Some outputs are missing. Read the error printed by the previous cell.")

PASS: The pipeline ran end to end and wrote all its outputs.


## 4. Test the pieces

Because each step is a function, you can test it with tiny inputs where you **know** the right answer. This is
how you catch a bug before it is buried inside 1,000 rows. Import the script like a library:

### Exercise 7.3 [Level 1: fill the blanks]

In [7]:
import importlib
import pipeline
importlib.reload(pipeline)          # pick up any edits you made to pipeline.py

print(pipeline.to_number(pd.Series(["€1,250.00", "3", None])).tolist())
print(pipeline.to_number(pd.Series(["1.250,00", "2,5"]), decimal=",").tolist())
print(pipeline.normalise_asset_id(pd.Series([" cnc001", "CMP-0O1"])).tolist())

[1250.0, 3.0, <NA>]
[1250.0, 2.5]
['CNC-001', 'CMP-0O1']


In [8]:
a = pipeline.to_number(pd.Series(["€1,250.00", "3"])).tolist()
b = pipeline.to_number(pd.Series(["1.250,00", "2,5"]), decimal=",").tolist()
c = pipeline.normalise_asset_id(pd.Series([" cnc001"])).tolist()
check(a == [1250.0, 3.0] and b == [1250.0, 2.5] and c == ["CNC-001"],
      "Helpers behave on known inputs. Note 'CMP-0O1' is left alone: a letter O is not a digit, so it stays visible as unknown.",
      "One of the helpers returned something unexpected.")

PASS: Helpers behave on known inputs. Note 'CMP-0O1' is left alone: a letter O is not a digit, so it stays visible as unknown.


## 5. Change a setting, not the code

The maintenance manager thinks 80 hours is too generous and wants any job over one 12-hour shift flagged. Copy the config, change the threshold and
the run name, run again, compare.

### Exercise 7.4 [Level 1: fill the blanks]

In [9]:
strict = json.loads(json.dumps(CONFIG))          # a full independent copy of the dictionary
strict["run_name"] = "cmms_strict"
strict["thresholds"]["max_hours"] = 12
with open("config_strict.json", "w", encoding="utf-8") as f:
    json.dump(strict, f, indent=2, ensure_ascii=False)

!{sys.executable} pipeline.py config_strict.json

2026-10-06 14:14:38,778 INFO    Run 'cmms_strict' started
2026-10-06 14:14:38,783 INFO    Loaded 1016 rows from data/work_orders_cmms.csv


2026-10-06 14:14:38,805 INFO    Dedup: 18 exact duplicates, 25 older versions removed, 973 work orders


2026-10-06 14:14:39,030 WARNING Rule closed_before_raised: 6 rows
2026-10-06 14:14:39,030 WARNING Rule negative_hours: 4 rows
2026-10-06 14:14:39,031 WARNING Rule implausible_hours: 11 rows
2026-10-06 14:14:39,031 WARNING Rule closed_without_date: 5 rows
2026-10-06 14:14:39,031 WARNING Rule unknown_asset: 6 rows
2026-10-06 14:14:39,032 INFO    Rule negative_cost: 1 rows
2026-10-06 14:14:39,032 INFO    Rule missing_hours: 33 rows
2026-10-06 14:14:39,033 INFO    Rule missing_cost: 88 rows


2026-10-06 14:14:39,080 INFO    Outputs written to outputs/cmms_strict
2026-10-06 14:14:39,081 INFO    Finished: {'rows_out': 973, 'errors': 32, 'output_dir': 'outputs/cmms_strict'}


In [10]:
base = pd.read_csv("outputs/cmms_full_history/dq_report.csv").set_index("rule")["rows_flagged"]
new = pd.read_csv("outputs/cmms_strict/dq_report.csv").set_index("rule")["rows_flagged"]
pd.DataFrame({"80 h": base, "12 h": new})

,80 h,12 h
rule,,
closed_before_raised,6,6
negative_hours,4,4
implausible_hours,3,11
closed_without_date,5,5
unknown_asset,6,6
implausible_cost,0,0
negative_cost,1,1
missing_hours,33,33
missing_cost,88,88


More jobs flagged. Whether 12 hours is right is an engineering conversation. What the pipeline gives you is the
ability to have that conversation with numbers, and to rerun it in seconds.

## 6. Running it outside Colab

On a laptop or server with Python installed, the same two files work:

```
pip install pandas openpyxl
python pipeline.py config_cmms.json
```

Schedule that line (Windows Task Scheduler, cron, or your data platform) and the monthly clean-up runs itself.
Keep `pipeline.py` and the configs in version control (Git) so every change is recorded.

## 7. Stretch [Level 3: Gemini allowed]  Review AI code like a pull request

Ask Gemini to **add one feature** to `pipeline.py`, for example: *"Add a function that writes a single Excel file
`summary.xlsx` to the run folder with two sheets: the dq_report, and a count of work orders by site and wo_type
for KPI-ready rows."*

Before you accept it, review it like a colleague's code:

* Does it change anything it was not asked to change?
* Does it hard-code anything that should be in the config (paths, thresholds)?
* Does it handle a run where there are zero KPI-ready rows?
* Did it add a library that is not installed?
* Run it. Open the output. Do the numbers match `dq_report.csv`?

## Summary

* Functions for each step, a config file for everything that varies, one command to run it.
* `%%writefile` saves a cell as a file. `!python script.py` runs it.
* Test helper functions on tiny inputs with known answers.
* Hard stops (`raise ValueError(...)`) with a message that tells the next person exactly what to fix.
* A log file and a run folder per run give you an audit trail.

Next: **08 Mini-Capstone**.